# Three-sensor pilot: NOAA-20 geolocation plan

Extend the July 6, 2025 Cameroon Q3 candidate neighborhood from Aqua + NOAA-21 to Aqua + NOAA-20 + NOAA-21. This notebook selects the existing Aqua and NOAA-20 sample observations using their actual dates/coordinates, verifies that they meet the project's recorded 1 km / 3 h candidate rule, extracts the exact geolocation filename from the NOAA-20 L2 input metadata, and resolves that exact filename through CMR. It writes a bounded one-file plan; it does not download data.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/reference').is_dir())
sys.path.insert(0, str(ROOT))
from src.firewatch.geolocation_manifest import extract_geolocation_reference, query_exact_geolocation_manifest
sample_path = ROOT / 'data/processed/source_pixel_validation_2025_stratified.csv'
l2_dir = ROOT / 'data/reference/granules/validation_2025'
out_path = ROOT / 'data/reference/geolocation_coverage_plan_2025_noaa20.csv'
sample = pd.read_csv(sample_path)
display(sample.groupby(['quarter','satellite','candidate_status_label']).size().rename('sample_rows').reset_index().tail(12))

,quarter,satellite,candidate_status_label,sample_rows
27,Q3,SNPP,no_candidate_link_under_1km_3h,1
28,Q3,Terra,no_candidate_link_under_1km_3h,1
29,Q4,Aqua,has_candidate_link,1
30,Q4,Aqua,no_candidate_link_under_1km_3h,1
31,Q4,N20,has_candidate_link,1
32,Q4,N20,no_candidate_link_under_1km_3h,1
33,Q4,N21,has_candidate_link,1
34,Q4,N21,no_candidate_link_under_1km_3h,1
35,Q4,SNPP,has_candidate_link,1
36,Q4,SNPP,no_candidate_link_under_1km_3h,1


In [2]:
def great_circle_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    a1, a2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = a2-a1, np.radians(lon2-lon1)
    h = np.sin(dlat/2)**2 + np.cos(a1)*np.cos(a2)*np.sin(dlon/2)**2
    return float(2*r*np.arcsin(np.sqrt(np.clip(h,0,1))))
q3_linked = sample[(sample.quarter == 'Q3') & sample.candidate_status & sample.candidate_status_label.eq('has_candidate_link')]
aqua = q3_linked[q3_linked.satellite.eq('Aqua')].iloc[0]
n20 = q3_linked[q3_linked.satellite.eq('N20')].iloc[0]
delta_min = abs((pd.Timestamp(aqua.observation_time_utc)-pd.Timestamp(n20.observation_time_utc)).total_seconds())/60
distance_km = great_circle_km(aqua.csv_latitude,aqua.csv_longitude,n20.csv_latitude,n20.csv_longitude)
assert delta_min <= 180 and distance_km <= 1.0, 'Selected sample rows fail the configured candidate thresholds'
n20_l2 = l2_dir / n20.granule_id
assert n20_l2.is_file(), f'Missing downloaded source L2 granule: {n20_l2}'
display(pd.DataFrame([{'platform':r.satellite,'observation_utc':r.observation_time_utc,'latitude':r.csv_latitude,'longitude':r.csv_longitude,'L2_granule':r.granule_id} for r in [aqua,n20]]))
print(f'Pair distance: {distance_km:.6f} km; time separation: {delta_min:.0f} min')

,platform,observation_utc,latitude,longitude,L2_granule
0,Aqua,2025-07-06T13:48:00+00:00,11.835,14.99200,MYD14.A2025187.1345.061.2025188110859.hdf
1,N20,2025-07-06T11:39:00+00:00,11.843,14.99202,VJ114IMG.A2025187.1136.002.2025187162721.nc


Pair distance: 0.889563 km; time separation: 129 min


In [3]:
reference = extract_geolocation_reference(n20_l2)
reference.update({
    'association_context': 'Aqua–NOAA-20 Q3 candidate neighborhood, selected from the stratified sample',
    'candidate_link_distance_km': distance_km,
    'candidate_link_time_delta_minutes': delta_min,
    'observation_id': n20.observation_id,
})
plan_rows, metadata = query_exact_geolocation_manifest([reference])
plan = pd.DataFrame(plan_rows)
plan.to_csv(out_path, index=False)
out_path.with_suffix('.metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
display(plan[['source_l2_filename','geolocation_short_name','geolocation_granule_id','cmr_collection_version','granule_size_cmr_mb','data_url']])
print(f'One exact geolocation file; CMR size estimate {plan.granule_size_cmr_mb.sum():.3f} MiB; no data downloaded.')

,source_l2_filename,geolocation_short_name,geolocation_granule_id,cmr_collection_version,granule_size_cmr_mb,data_url
0,VJ114IMG.A2025187.1136.002.2025187162721.nc,VJ103IMG,VJ103IMG.A2025187.1136.021.2025187154117.nc,2.1,177.332589,https://data.laadsdaac.earthdatacloud.nasa.gov...


One exact geolocation file; CMR size estimate 177.333 MiB; no data downloaded.


## Selection limits

This is not an independent validation sample: both observations were selected because they already met the candidate rule. The three-sensor extension adds NOAA-20's exact native geolocation companion so that its source mask can be compared with both other sensors at these nearby coordinates. The overpass separations differ (Aqua–NOAA-21: 78 minutes; Aqua–NOAA-20: 129 minutes), so masks remain time-specific. The result is a sensor harmonization case study, not detection accuracy or continuous-fire evidence.